# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [3]:
import os
from glob import glob

# Write your code below.
PRICE_DATA = os.getenv("PRICE_DATA")  
# error check if not sure about data, 
if not PRICE_DATA:
    raise ValueError("PRICE_DATA is not defined in the environment.")

# search function.
parquet_files = sorted(
    glob(os.path.join(PRICE_DATA, "**", "*.parquet"), recursive=True)
)
# same as above, the error check !!
if not parquet_files:
    raise FileNotFoundError(f"No parquet files found in {PRICE_DATA}")

print("PRICE_DATA:", PRICE_DATA)
print("Parquet files found:", len(parquet_files))

PRICE_DATA: ../../05_src/data/prices/
Parquet files found: 3146


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

Part 1: Load the data and add lags for each ticker

In [8]:
import pandas as pd
dd_prices = dd.read_parquet(parquet_files)

if "Adj Close" in dd_prices.columns:
    dd_prices = dd_prices.rename(columns={"Adj Close": "Adj_Close"}) 

dd_prices = dd_prices.set_index("ticker")


def add_lags(ticker_data):
    # Sort observations 
    ticker_data = ticker_data.sort_values("Date")

    return ticker_data.assign(
        Close_lag_1=ticker_data["Close"].shift(1),
        Adj_Close_lag_1=ticker_data["Adj_Close"].shift(1)
    )

# Define output types.
meta = dd_prices._meta.assign(
    Close_lag_1=pd.Series(dtype="float64"),
    Adj_Close_lag_1=pd.Series(dtype="float64")
)

dd_lags = (
    dd_prices.groupby("ticker", group_keys=False)
    .apply(add_lags, meta=meta)
)

Part 2: Add lags for variables Close and Adj_Close.:: returns: (Close / Close_lag_1) - 1

In [9]:
dd_returns = dd_lags.assign(
    returns=lambda x: (x["Close"] / x["Close_lag_1"]) - 1
)

Part 3: Add the following range:  hi_lo_range: this is the day's High minus Low.

In [10]:
dd_range = dd_returns.assign(
    hi_lo_range=lambda x: x["High"] - x["Low"]
)

Assign the result to dd_feat.

In [11]:
dd_feat = dd_range
dd_feat

,Date,Open,High,Low,Close,Adj_Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
npartitions=90,,,,,,,,,,,,,
ACN,datetime64[us],float64,float64,float64,float64,float64,float64,string,int32,float64,float64,float64,float64
AKR,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

Part 1: Convert the Dask DataFrame to pandas

In [13]:
df_feat = dd_feat.compute()

# Make ticker column & sort by date
df_feat = (
    df_feat.reset_index()
    .sort_values(["ticker", "Date"])
    .reset_index(drop=True)
)

df_feat.head()

,ticker,Date,Open,High,Low,Close,Adj_Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
0,ACN,2001-07-19,15.10,15.29,15.00,15.17,11.404394,34994300.0,ACN.csv,2001,NaN,NaN,NaN,0.29
1,ACN,2001-07-20,15.05,15.05,14.80,15.01,11.284108,9238500.0,ACN.csv,2001,15.17,11.404394,-0.010547,0.25
2,ACN,2001-07-23,15.00,15.01,14.55,15.00,11.276587,7501000.0,ACN.csv,2001,15.01,11.284108,-0.000666,0.46
3,ACN,2001-07-24,14.95,14.97,14.70,14.86,11.171341,3537300.0,ACN.csv,2001,15.00,11.276587,-0.009333,0.27
4,ACN,2001-07-25,14.70,14.95,14.65,14.95,11.238999,4208100.0,ACN.csv,2001,14.86,11.171341,0.006057,0.30


Part 2: Add a new feature containing the moving average of returns using a window of 10 days. There are several ways to solve this task, a simple one uses .rolling(10).mean().

In [14]:
# moving average
df_feat["returns_ma_10"] = (
    df_feat.groupby("ticker")["returns"]
    .transform(lambda x: x.rolling(10).mean())
)

df_feat[["ticker", "Date", "returns", "returns_ma_10"]].head(15)

,ticker,Date,returns,returns_ma_10
0,ACN,2001-07-19,NaN,NaN
1,ACN,2001-07-20,-0.010547,NaN
2,ACN,2001-07-23,-0.000666,NaN
3,ACN,2001-07-24,-0.009333,NaN
4,ACN,2001-07-25,0.006057,NaN
5,ACN,2001-07-26,-0.030100,NaN
6,ACN,2001-07-27,0.000690,NaN
7,ACN,2001-07-30,0.013094,NaN
8,ACN,2001-07-31,0.017687,NaN
9,ACN,2001-08-01,0.036096,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
I dont think so it was very necessary, I belive this could also be done in DASK, but we did it as it was asked. But pandas is simple and faster.

+ Would it have been better to do it in Dask? Why?
I dont think it would be better using DAsk over the Pandas for this particular data, in my opnion dask may took more pocessing; but for a larger data set, it will definately be preferale choice as it takes partitions of data sets and process 


(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.